# Notebook 00: Setup & Data Loading

**OP'26 Analytics — Agentic AI Dynamic Tariff Optimization for EV Charging**

This notebook verifies the environment and loads both datasets:
- **ACN-Data**: 30k+ Caltech/JPL charging sessions (Apr–Dec 2018)
- **UrbanEV (ST-EVCDP)**: Shenzhen, 5-min interval data across ~247 district grids (Jun–Dec 2022)

All subsequent notebooks import from `src/` helpers built here.

In [1]:
import sys
from pathlib import Path

# Ensure src/ is on the path
SRC = Path('..') / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import sklearn
import seaborn as sns

print(f'pandas  : {pd.__version__}')
print(f'numpy   : {np.__version__}')
print(f'sklearn : {sklearn.__version__}')
print(f'matplotlib: {matplotlib.__version__}')

pandas  : 3.0.3
numpy   : 2.4.6
sklearn : 1.9.0
matplotlib: 3.10.9


## 1. Load ACN-Data

In [2]:
from io_utils import load_acn

acn_raw = load_acn()
print(f'ACN shape: {acn_raw.shape}')
print(f'Columns  : {list(acn_raw.columns)}')
acn_raw.head(3)

ACN shape: (16304, 26)
Columns  : ['_meta', 'end', 'min_kWh', 'site', 'start', '_items', '_id', 'clusterID', 'connectionTime', 'disconnectTime', 'doneChargingTime', 'kWhDelivered', 'sessionID', 'siteID', 'spaceID', 'stationID', 'timezone', 'userID', 'WhPerMile', 'kWhRequested', 'milesRequested', 'minutesAvailable', 'modifiedAt', 'paymentRequired', 'requestedDeparture', 'userID.1']


,_meta,end,min_kWh,site,start,_items,_id,clusterID,connectionTime,disconnectTime,...,timezone,userID,WhPerMile,kWhRequested,milesRequested,minutesAvailable,modifiedAt,paymentRequired,requestedDeparture,userID.1
0,NaN,NaN,NaN,caltech,NaN,NaN,5bc90cb9f9af8b0d7fe77cd2,39.0,2018-04-25 11:08:04+00:00,2018-04-25 13:20:10+00:00,...,America/Los_Angeles,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,5bc90cb9f9af8b0d7fe77cd3,39.0,2018-04-25 13:45:10+00:00,2018-04-26 00:56:16+00:00,...,America/Los_Angeles,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,5bc90cb9f9af8b0d7fe77cd4,39.0,2018-04-25 13:45:50+00:00,2018-04-25 23:04:45+00:00,...,America/Los_Angeles,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# Valid session rows (non-null kWh)
acn = acn_raw[acn_raw['kWhDelivered'].notna() & (acn_raw['kWhDelivered'] > 0)].copy()
print(f'Valid sessions: {len(acn):,}')
print(f'Date range   : {acn["connectionTime"].min()} → {acn["connectionTime"].max()}')
print(f'Sites        : {acn["siteID"].unique()}')
acn[['kWhDelivered','connectionTime','disconnectTime','doneChargingTime','stationID']].describe()

Valid sessions: 14,999
Date range   : 2018-04-25 11:08:04+00:00 → 2018-12-16 03:01:46+00:00
Sites        : [2.]


,kWhDelivered
count,14999.000000
mean,9.002466
std,7.055848
min,0.501000
25%,4.008500
50%,7.435000
75%,13.204000
max,69.373000


## 2. Load UrbanEV Panel

In [4]:
from io_utils import load_urbanev_panel, load_urbanev_info

uev = load_urbanev_panel()
print(f'UrbanEV shape: {uev.shape}')
print(f'Columns: {list(uev.columns)}')
print(f'Grids: {uev["grid_id"].nunique()}')
print(f'Timesteps: {uev["timestamp"].nunique()}')
print(f'Date range: {uev["datetime"].min()} → {uev["datetime"].max()}')
uev.head(3)

UrbanEV shape: (2134080, 15)
Columns: ['timestamp', 'grid_id', 'occupancy', 'volume', 'duration', 'price', 'datetime', 'count', 'fast_count', 'slow_count', 'area', 'lon', 'la', 'CBD', 'dynamic_pricing']
Grids: 247
Timesteps: 8640
Date range: 2022-06-19 00:00:00 → 2022-07-18 23:55:00


,timestamp,grid_id,occupancy,volume,duration,price,datetime,count,fast_count,slow_count,area,lon,la,CBD,dynamic_pricing
0,1,1000,60,16.197222,3.157778,0.894267,2022-06-19 00:00:00,193,0,193,3.08,114.2695,22.72019,0,0
1,2,1000,60,24.791667,4.833333,0.894267,2022-06-19 00:05:00,193,0,193,3.08,114.2695,22.72019,0,0
2,3,1000,60,24.791667,4.833333,0.894267,2022-06-19 00:10:00,193,0,193,3.08,114.2695,22.72019,0,0


In [5]:
print('=== UrbanEV key stats ===')
print(uev[['occupancy','volume','duration','price']].describe().round(3))
print(f'\nGrids with dynamic_pricing=1: {uev["dynamic_pricing"].sum() // uev["timestamp"].nunique()}')
print(f'CBD grids: {uev["CBD"].sum() // uev["timestamp"].nunique()}')

=== UrbanEV key stats ===
         occupancy       volume     duration        price
count  2134080.000  2134080.000  2134080.000  2134080.000
mean        21.902       36.678        1.473        0.959
std         24.800      101.893        1.803        0.181
min          0.000        0.000        0.000        0.250
25%          5.000        2.333        0.333        0.850
50%         13.000        7.000        0.833        0.984
75%         30.000       23.333        1.862        1.073
max        220.000     1492.500       17.083        1.470

Grids with dynamic_pricing=1: 57
CBD grids: 62


## 3. Cache to Parquet

Cache processed copies so later notebooks skip the xlsx/csv parse step.

In [6]:
OUT = Path('..') / 'data_processed'
OUT.mkdir(exist_ok=True)

# ACN — save only valid sessions, stringify tz-aware timestamps for parquet compat
acn_save = acn.copy()
for col in ['connectionTime','disconnectTime','doneChargingTime']:
    if col in acn_save.columns:
        acn_save[col] = acn_save[col].astype(str)
acn_save.to_parquet(OUT / 'acn_raw.parquet', index=False)

uev.to_parquet(OUT / 'urbanev_panel.parquet', index=False)
print('Cached to data_processed/')
print(f'  acn_raw.parquet      : {(OUT/"acn_raw.parquet").stat().st_size/1e6:.1f} MB')
print(f'  urbanev_panel.parquet: {(OUT/"urbanev_panel.parquet").stat().st_size/1e6:.1f} MB')

Cached to data_processed/
  acn_raw.parquet      : 1.0 MB
  urbanev_panel.parquet: 12.8 MB
